# smolagents Teaching Tutorial: 10 Applications (Basic → Intermediate)

Welcome! This notebook is a hands-on, build-as-you-learn tutorial for [smolagents](https://github.com/huggingface/smolagents) — Hugging Face's minimalist library for building agents that *think in code*.

## What you will learn
By the end of this notebook you will have built **10 progressively harder agent applications**:

| #  | App | Concept Introduced |
|----|---------------------------------------|---------------------------------|
| 1  | Hello Agent                           | `CodeAgent` basics              |
| 2  | Calculator Agent                      | Built-in Python execution       |
| 3  | Web Search Agent                      | Using built-in `WebSearchTool`  |
| 4  | Custom Tool — Word Counter            | `@tool` decorator               |
| 5  | Weather Agent (mock API)              | Tools that call functions       |
| 6  | Multi-Tool Research Assistant         | Combining tools                 |
| 7  | File Q&A Agent                        | Reading local files as a tool   |
| 8  | Wikipedia Summarizer Agent            | `ToolCallingAgent` + real API   |
| 9  | Multi-Agent System (Manager + Worker) | Agent orchestration             |
| 10 | Mini Data Analyst Agent               | Pandas + plotting in an agent   |

## Prerequisites
- Python 3.10+
- A Hugging Face account (free) — we'll help you get a token in the next section.

> **Note**: smolagents works with many model backends (HF Inference API, OpenAI, Anthropic, local models via Transformers, Ollama, etc.). For this tutorial we default to the **Hugging Face Inference API** because it's free and easy to set up.

## 0. Setup

### 0.1 Install smolagents

In [13]:
# Install smolagents and a few helpers used throughout the notebook
%pip install -q "smolagents[toolkit]" pandas matplotlib wikipedia-api requests


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### 0.2 Get a Hugging Face token (one-time setup)

smolagents needs an LLM to drive its reasoning. The simplest free option is the **Hugging Face Inference API**, which requires a token.

**Step-by-step:**
1. Create a free account at <https://huggingface.co/join> (skip if you have one).
2. Open <https://huggingface.co/settings/tokens>.
3. Click **"Create new token"**.
4. Give it a name (e.g. `smolagents-tutorial`).
5. Token type: **Read** is enough. If you plan to use models requiring inference providers, also enable **"Make calls to inference providers"** under fine-grained permissions.
6. Click **Create token** and **copy** the token (it starts with `hf_...`).
7. **Paste it into the cell below** when prompted (it's hidden as you type).

> 🔐 The token is only stored in this kernel's memory — it is **not** written to disk by this notebook.

In [14]:
import os
import getpass
from huggingface_hub import HfApi

# Set FORCE_NEW_TOKEN = True if your token expired and you want to re-enter it
FORCE_NEW_TOKEN = True

if FORCE_NEW_TOKEN or not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("Paste your Hugging Face token (hf_...): ").strip()

assert os.environ["HF_TOKEN"].startswith("hf_"), "That doesn't look like a HF token — it should start with 'hf_'."

# Verify the token is actually valid (catches expired tokens BEFORE you hit App 1)
try:
    user = HfApi().whoami(token=os.environ["HF_TOKEN"])
    print(f"✓ HF_TOKEN is valid. Logged in as: {user['name']}")
except Exception as e:
    raise RuntimeError(
        "❌ Your HF token is invalid or expired.\n"
        "→ Create a fresh one at https://huggingface.co/settings/tokens\n"
        "  (type: Read, and enable 'Make calls to inference providers').\n"
        "→ Then re-run this cell."
    ) from e

✓ HF_TOKEN is valid. Logged in as: lcabanillas


### 0.3 Pick a model

We'll use `InferenceClientModel`, which talks to Hugging Face's hosted inference. The default model below is a small, capable open model. You can swap it any time.

In [17]:
from smolagents import InferenceClientModel

# Widely-accessible default. Swap if you have access to something stronger
# (e.g. "Qwen/Qwen2.5-Coder-32B-Instruct", "mistralai/Mistral-7B-Instruct-v0.3").
MODEL_ID = "Qwen/Qwen2.5-72B-Instruct"

model = InferenceClientModel(model_id=MODEL_ID, token=os.environ["HF_TOKEN"])
print(f"✓ Model wired: {MODEL_ID}")

✓ Model wired: Qwen/Qwen2.5-72B-Instruct


> 💡 **Don't have HF inference access for that model?** Try `"meta-llama/Llama-3.2-3B-Instruct"`, `"HuggingFaceH4/zephyr-7b-beta"`, or any model you have access to. You can also use `LiteLLMModel` to point at OpenAI, Anthropic, Ollama, etc.

---
## App 1 — Hello Agent 👋

**Concept:** The simplest possible agent. A `CodeAgent` writes Python code to solve your task, executes it, and returns the result.

We pass an **empty tool list** — the agent only has Python itself.

In [18]:
from smolagents import CodeAgent

hello_agent = CodeAgent(tools=[], model=model)

hello_agent.run("Say hello to a student named Alex and wish them luck on their agentic AI course.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Say hello to a student named Alex and wish them luck on their agentic AI course.                                │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-72B-Instruct ──────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  print("Hello Alex! I hope you're doing well. Wishing you all the best in your agentic AI course!")               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Hello Alex! I hope you're doing well. Wishing you all the best in your agentic AI course!

Out: None

[Step 1: Duration 2.30 seconds| Input tokens: 1,954 | Output tokens: 55]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Hello Alex! I hope you're doing well. Wishing you all the best in your agentic AI course!")        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Hello Alex! I hope you're doing well. Wishing you all the best in your agentic AI course!

[Step 2: Duration 2.30 seconds| Input tokens: 4,070 | Output tokens: 113]

"Hello Alex! I hope you're doing well. Wishing you all the best in your agentic AI course!"

🔎 **What just happened?**
- The agent received your task.
- It generated a small Python snippet (e.g. a `print(...)` or a `final_answer(...)`).
- The snippet ran inside a sandboxed Python executor.
- The output was returned.

---
## App 2 — Calculator Agent 🧮

**Concept:** Even without tools, a `CodeAgent` can do real math because it can write and run Python.

In [19]:
calc_agent = CodeAgent(tools=[], model=model, additional_authorized_imports=["math"])

calc_agent.run(
    "If a loan of $12,500 has 6.4% annual interest compounded monthly, "
    "what is the balance after 3 years? Show the formula you used."
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ If a loan of $12,500 has 6.4% annual interest compounded monthly, what is the balance after 3 years? Show the   │
│ formula you used.                                                                                               │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-72B-Instruct ──────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Define the variables                                                                                           
  P = 12500  # Principal amount                                                                                    
  r = 6.4 / 100  # Annual interest rate (as a decimal)                                                             
  n = 12  # Number of times interest is compounded per year                                                        
  t = 3  # Time in years                                                                                           
                                                                                                                   
  # Calculate the balance after 3 years                                                                            
  A = P * (1 + r / n) ** (n * t)                                                                                   
  print(A)                                                                                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
15138.156203997969

Out: None

[Step 1: Duration 6.66 seconds| Input tokens: 1,973 | Output tokens: 221]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  formula = "A = P * (1 + r / n) ** (n * t)"                                                                       
  final_answer(formula + ", where A is the balance after 3 years, P is the principal amount ($12,500), r is the    
  annual interest rate (6.4%), n is the number of times interest is compounded per year (12), and t is the time    
  the money is borrowed for, in years (3). The balance after 3 years is approximately $15,138.16.")                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: A = P * (1 + r / n) ** (n * t), where A is the balance after 3 years, P is the principal amount 
($12,500), r is the annual interest rate (6.4%), n is the number of times interest is compounded per year (12), and
t is the time the money is borrowed for, in years (3). The balance after 3 years is approximately $15,138.16.

[Step 2: Duration 4.83 seconds| Input tokens: 4,339 | Output tokens: 369]

'A = P * (1 + r / n) ** (n * t), where A is the balance after 3 years, P is the principal amount ($12,500), r is the annual interest rate (6.4%), n is the number of times interest is compounded per year (12), and t is the time the money is borrowed for, in years (3). The balance after 3 years is approximately $15,138.16.'

🔎 Note the `additional_authorized_imports=["math"]` — by default, the executor only allows a safe subset of imports. You explicitly opt-in to what the agent can use.

---
## App 3 — Web Search Agent 🌐

**Concept:** Give the agent a real tool. `WebSearchTool` ships with smolagents and uses DuckDuckGo under the hood — no API key needed.

In [20]:
from smolagents import CodeAgent, WebSearchTool

search_agent = CodeAgent(tools=[WebSearchTool()], model=model)

search_agent.run("Find the current population of Reykjavik, Iceland and tell me the source.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find the current population of Reykjavik, Iceland and tell me the source.                                       │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-72B-Instruct ──────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  search_results = web_search(query="current population of Reykjavik, Iceland")                                    
  print(search_results)                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[Reykjavik Population 2026](https://worldpopulationreview.com/cities/iceland/reykjavik)
 Reykjavik Demographics Reykjavik is the largest settlement in Iceland , with people from over 100 countries 
calling the city home. Poles, Danes and Lithuanians are the most common ethnic minorities living in Reykjavik, and 
in 2009 as much as 8% of the population was made up of non-natives.

[Reykjavík Population 2026 — 138,772 People | Growth & Area](https://citypopulationdata.com/city/reykjavik-iceland)
The population  of  Reykjavík,  Iceland is 138,772 in 2026. Explore live stats, growth rate, population density, 
area size, and see how Reykjavík ranks among other cities in Iceland .

[Reykjavík - Wikipedia](https://en.wikipedia.org/wiki/Reykjavík)
Reykjavík[a] is the capital and largest city of Iceland . It is located on the southern shore of the Faxaflói bay 
in southwest Iceland and has a latitude of 64°08′ N, making it the world's northernmost capital of a sovereign 
state. [b] Reykjavík has a population  of around 139,000 as of 2025, [8] and the surrounding Capital Region has a 
population  of around 249,000, constituting ...

[Greater Reykjavik (Iceland): Settlements - Population Statistics 
...](https://www.citypopulation.de/en/iceland/reykjavik/)
Settlements The population  of the settlements of Greater Reykjavik (Stór-Reykjavík) according to official 
estimates. The icon links to further information about a selected division including its population structure 
(gender, age groups, age distribution).

[Population Reykjavik (Iceland), number, employment, unemployment 
...](https://bdeex.com/en-gb/naselenie/iceland/reykjavik/)
The population page for Reykjavik,  Iceland brings together city-level demographic figures in a simple, 
comparison-friendly structure. Start with the main population data for Reykjavik and then move across related pages
to compare cities and country-level indicators in Iceland .

[Iceland Population: Demographics & Society Overview 2026](https://www.iceland.org/population)
Overview of Iceland's  population (~383,000): demographics, where people live, the patronymic naming system, 
Íslendingabók genealogy database.

[Iceland Cities by Population 2026](https://worldpopulationreview.com/cities/iceland)
 Iceland Overview The largest city in Iceland is Reykjavik, with a population  of 249,228 people. Iceland is the 
108th largest country in the world by area, and has a population  of about 402,000 as of 2026. Reykjavik is the 
largest Icelandic city, contributing over 249.2K people to the total population  of the country.

[Population - key figures 1703-2026 - 
PxWeb](https://px.hagstofa.is/pxen/pxweb/en/Ibuar/Ibuar__mannfjoldi__1_yfirlit__yfirlit_mannfjolda/MAN00000.px)
 Population - key figures 1703-2026 2026 2025 2024 2023 2022 2021 2020 2019 2018 2017 2016 2015 2014 2013 2012 2011
2010 2009 2008 2007 2006 2005 2004 2003 2002 2001 2000 1999 1998 1997 1996 1995 1994 1993 1992 1991 1990 1989 1988 
1987 1986 1985 1984 1983 1982 1981 1980 1979 1978 1977 1976 1975 1974 1973 1972 1971 1970 1969 1968 1967 1966 1965 
1964 1963 1962 1961 1960 1959 1958 1957 1956 1955 ...

[Reykjavík - Population Trends and Demographics - CityFacts](https://www.city-facts.com/reykjavik/population)
 Reykjavík is the capital and largest city of Iceland . It is located in southwestern Iceland , on the southern 
shore of Faxa Bay. Its latitude is 64°08' N, making it the world's northernmost capital of a sovereign state. With 
a population  of around 128,793, it is the heart of Iceland's cultural, economic and governmental activity, and is 
a popular tourist destination. Source: Wikipedia

[Population by urban nuclei and localites 2025 - Statistics 
Iceland](https://statice.is/publications/news-archive/inhabitants/population-by-urban-nuclei-and-localites-2025/)
On January 1, 2025, a total of 244,536 people lived in the Greater Reykjavík area, i.e., the continuous settlement 
stretching from Hafnarfjörður 

[Step 1: Duration 3.49 seconds| Input tokens: 2,007 | Output tokens: 57]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("The current population of Reykjavik, Iceland is 138,772. The source is                             
  [CityPopulationData](https://citypopulationdata.com/city/reykjavik-iceland).")                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: The current population of Reykjavik, Iceland is 138,772. The source is 
[CityPopulationData](https://citypopulationdata.com/city/reykjavik-iceland).

[Step 2: Duration 4.47 seconds| Input tokens: 5,547 | Output tokens: 186]

'The current population of Reykjavik, Iceland is 138,772. The source is [CityPopulationData](https://citypopulationdata.com/city/reykjavik-iceland).'

---
## App 4 — Custom Tool: Word Counter ✍️

**Concept:** Build your own tool. Any Python function decorated with `@tool` and a clear docstring becomes callable by the agent.

**Rules of thumb for tools:**
- Type-annotate every argument.
- Write a docstring describing *what it does* and *each argument* — the agent reads this.
- Keep them small and composable.

In [21]:
from smolagents import CodeAgent, tool

@tool
def word_stats(text: str) -> dict:
    """Returns word count, character count, and average word length for a piece of text.

    Args:
        text: The text to analyze.
    """
    words = text.split()
    n_words = len(words)
    n_chars = len(text)
    avg_len = (sum(len(w) for w in words) / n_words) if n_words else 0
    return {"words": n_words, "chars": n_chars, "avg_word_length": round(avg_len, 2)}

writer_agent = CodeAgent(tools=[word_stats], model=model)

writer_agent.run(
    "Analyze the following sentence and tell me if it's concise enough for a tweet (<= 280 chars): "
    "'Agentic AI lets models plan, use tools, and act — not just chat.'"
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Analyze the following sentence and tell me if it's concise enough for a tweet (<= 280 chars): 'Agentic AI lets  │
│ models plan, use tools, and act — not just chat.'                                                               │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-72B-Instruct ──────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  sentence = "Agentic AI lets models plan, use tools, and act — not just chat."                                    
  _, char_count, _ = word_stats(sentence)                                                                          
  print(f"Character count: {char_count}")                                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Character count: chars

Out: None

[Step 1: Duration 3.15 seconds| Input tokens: 2,020 | Output tokens: 86]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(is_concise)                                                                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: True

[Step 3: Duration 2.12 seconds| Input tokens: 6,720 | Output tokens: 254]

True

---
## App 5 — Weather Agent (mock API) ☀️

**Concept:** Tools can wrap any function — including ones that hit external services. To keep this tutorial dependency-free we use a deterministic *mock* weather service. Swap in a real API (OpenWeatherMap, etc.) for a real app.

In [22]:
import random
from smolagents import CodeAgent, tool

@tool
def get_weather(city: str, units: str = "celsius") -> str:
    """Returns a (mock) current weather report for a given city.

    Args:
        city: The city name, e.g. 'Baltimore'.
        units: 'celsius' or 'fahrenheit'.
    """
    rng = random.Random(hash(city) & 0xFFFF)
    temp_c = rng.randint(-5, 32)
    temp = temp_c if units == "celsius" else round(temp_c * 9 / 5 + 32)
    unit_label = "°C" if units == "celsius" else "°F"
    condition = rng.choice(["sunny", "cloudy", "rainy", "windy", "snowy"])
    return f"{city}: {temp}{unit_label}, {condition}."

weather_agent = CodeAgent(tools=[get_weather], model=model)

weather_agent.run(
    "Compare the weather today in Baltimore and Lima, and recommend what to wear in each. Use fahrenheit for Baltimore and celsius for Lima."
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Compare the weather today in Baltimore and Lima, and recommend what to wear in each. Use fahrenheit for         │
│ Baltimore and celsius for Lima.                                                                                 │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-72B-Instruct ──────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  baltimore_weather = get_weather(city='Baltimore', units='fahrenheit')                                            
  lima_weather = get_weather(city='Lima', units='celsius')                                                         
  print(f"Weather in Baltimore: {baltimore_weather}")                                                              
  print(f"Weather in Lima: {lima_weather}")                                                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Weather in Baltimore: Baltimore: 77°F, windy.
Weather in Lima: Lima: 8°C, sunny.

Out: None

[Step 1: Duration 3.04 seconds| Input tokens: 2,024 | Output tokens: 90]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(f"In Baltimore, it's recommended to wear: {baltimore_clothing}. In Lima, it's recommended to wear:  
  {lima_clothing}.")                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: In Baltimore, it's recommended to wear: light clothing. In Lima, it's recommended to wear: a light 
jacket.

[Step 3: Duration 4.01 seconds| Input tokens: 6,935 | Output tokens: 309]

"In Baltimore, it's recommended to wear: light clothing. In Lima, it's recommended to wear: a light jacket."

---
## App 6 — Multi-Tool Research Assistant 🔎

**Concept:** Real agents shine when given *several* tools and asked to combine them. Here we give the agent search + our word-stats tool, and ask it to research and then summarize concisely.

In [ ]:
from smolagents import CodeAgent, WebSearchTool

research_agent = CodeAgent(
    tools=[WebSearchTool(), word_stats],
    model=model,
)

research_agent.run(
    "Research what 'agentic RAG' means in 2024-2025, then write a 2-sentence summary. "
    "Use the word_stats tool to confirm the final summary is under 60 words."
)

---
## App 7 — File Q&A Agent 📄

**Concept:** Tools can read your local environment. We build a tiny notes file and give the agent a tool to query it.

In [ ]:
from pathlib import Path
from smolagents import CodeAgent, tool

NOTES_PATH = Path("course_notes.txt")
NOTES_PATH.write_text(
    """Week 6: Agentic RAG combines retrieval with tool-using agents.
Week 7: Evaluation focuses on faithfulness, groundedness, and tool-call accuracy.
Week 8: Ethics — covers responsible deployment, bias audits, and safety guardrails.
Project deadline: Project 3 is due 2026-07-26.
"""
)

@tool
def read_notes() -> str:
    """Returns the full text of the student's course notes file."""
    return NOTES_PATH.read_text()

notes_agent = CodeAgent(tools=[read_notes], model=model)

notes_agent.run("Using my course notes, when is Project 3 due and what does Week 8 cover?")

---
## App 8 — Wikipedia Summarizer (ToolCallingAgent) 📚

**Concept:** Two things new here:
1. We use the **real Wikipedia API** via the `wikipedia-api` library.
2. We use `ToolCallingAgent`, which uses the model's *native function-calling* format instead of writing Python code. It's a great fit for simple "call one tool, return the result" tasks.

In [ ]:
import wikipediaapi
from smolagents import ToolCallingAgent, tool

_wiki = wikipediaapi.Wikipedia(user_agent="smolagents-tutorial/1.0", language="en")

@tool
def wikipedia_summary(title: str, sentences: int = 3) -> str:
    """Fetches the lead summary of a Wikipedia article.

    Args:
        title: The article title, e.g. 'Ada Lovelace'.
        sentences: How many sentences from the article summary to return.
    """
    page = _wiki.page(title)
    if not page.exists():
        return f"No Wikipedia article titled '{title}'."
    summary = page.summary
    parts = summary.split(". ")
    return ". ".join(parts[:sentences]).strip() + ("." if not summary.endswith(".") else "")

wiki_agent = ToolCallingAgent(tools=[wikipedia_summary], model=model)

wiki_agent.run("Give me a 2-sentence summary of Ada Lovelace.")

🔎 **`CodeAgent` vs `ToolCallingAgent`:**
- `CodeAgent` is more flexible — it can chain logic in Python (`for`, `if`, math, etc.).
- `ToolCallingAgent` is simpler and more predictable for single-call use cases.

---
## App 9 — Multi-Agent System: Manager + Worker 🧑‍💼🛠️

**Concept:** smolagents lets you give one agent **other agents** as if they were tools (`managed_agents`). The manager plans; the worker executes a narrow job. This is the foundation of more sophisticated agentic systems.

In [ ]:
from smolagents import CodeAgent, ToolCallingAgent, WebSearchTool

# Worker: a small focused web searcher
web_worker = ToolCallingAgent(
    tools=[WebSearchTool()],
    model=model,
    name="web_worker",
    description="Searches the web and returns short factual answers with a source URL.",
    max_steps=4,
)

# Manager: delegates and synthesizes
manager = CodeAgent(
    tools=[],
    model=model,
    managed_agents=[web_worker],
)

manager.run(
    "Use the web_worker to find (a) who won the most recent FIFA World Cup and "
    "(b) the host country. Then compose a one-paragraph briefing for a sports newsletter."
)

🔎 **What's new:**
- The worker has its own tool set and prompt context.
- The manager *calls* the worker like a tool — passing it a sub-task in natural language.
- You can stack multiple specialized workers (researcher, coder, summarizer, …).

---
## App 10 — Mini Data Analyst 📊

**Concept:** Capstone — give a `CodeAgent` permission to use `pandas` and `matplotlib`, then ask it to analyze a small in-memory dataset and produce a chart. This is where code-writing agents really pay off.

In [ ]:
import pandas as pd
from smolagents import CodeAgent, tool

# A tiny sales dataset the agent can fetch
SALES = pd.DataFrame({
    "month": ["Jan", "Feb", "Mar", "Apr", "May", "Jun"],
    "product": ["A", "A", "A", "B", "B", "B"],
    "revenue": [1200, 1500, 1700, 900, 1300, 2100],
    "units":   [60,   72,   80,   30,   45,   70],
})

@tool
def get_sales_data() -> str:
    """Returns the sales dataset as CSV text (columns: month, product, revenue, units)."""
    return SALES.to_csv(index=False)

analyst_agent = CodeAgent(
    tools=[get_sales_data],
    model=model,
    additional_authorized_imports=["pandas", "io", "matplotlib", "matplotlib.pyplot"],
)

analyst_agent.run(
    "Load the sales data, compute total revenue per product, identify the best-selling product, "
    "and plot a bar chart of monthly revenue by product. Then give a 2-sentence executive summary."
)

---
## 🎓 Wrap-up

You just built **10 agents** going from a one-liner `hello` agent all the way to a multi-agent system and a mini data analyst.

### Key takeaways
- **`CodeAgent` writes Python** → flexible for analytical / multi-step tasks.
- **`ToolCallingAgent` calls one tool at a time** → predictable for narrow tasks.
- **Tools = annotated Python functions** with a clear docstring. That's it.
- **`managed_agents`** turns any agent into a tool for another agent — the building block of multi-agent systems.
- Always be explicit about **what imports** the executor is allowed to use (`additional_authorized_imports`).

### Suggested next steps
1. Swap `InferenceClientModel` for `LiteLLMModel` and try OpenAI / Anthropic / Ollama backends.
2. Replace the mock weather tool in App 5 with a real OpenWeatherMap call.
3. Extend App 9 with a *second* worker (e.g. a `summarizer` agent) and have the manager coordinate both.
4. Combine App 7 + App 10 into a personal-finance analyst that reads your own CSV.
5. Connect this notebook's ideas to **Week 8 — Responsible AI**: add a `guardrail` tool that refuses unsafe requests before the agent acts.

Happy building! 🧪🤖